In [3]:
import mysql.connector
import random
from datetime import datetime, timedelta
import pandas as pd

# ── Connect to MySQL ──────────────────────────────
conn = mysql.connector.connect(
    host     = "localhost",
    user     = "root",         # change if yours is different
    password = "Abhi@1445",  # your MySQL password
    database = "sirrus_tcg",
    autocommit=True
)
cur = conn.cursor()
random.seed(42)

# ── 1. INSERT CLIENTS ─────────────────────────────
clients = [
    (1,  'Fortune Developers',     'Bhopal', 'Rahul S',  '2024-01-10', 'Pro',1200000,'At Risk'),
    (2,  'Shashwat Realty',       'Ahmedabad', 'Priya M',  '2024-02-05', 'Pro',800000,'Churned'),
    (3,  'Rama Group',             'Pune',  'Vikram K', '2024-02-20', 'Basic',1800000,'Active'),
    (4,  'GERA Properties',        'Pune', 'Rahul S',  '2024-03-01', 'Pro',2400000,'Active'),
    (5,  'Chugh Group',            'Indore',   'Priya M',  '2024-03-15', 'Basic',1500000,'Churned'),
    (6,  'Prestige Group',         'Pune',  'Vikram K', '2024-04-01', 'Pro',1700000,'Active'),
    (7,  'Soumya Homes',           'Bhopal', 'Rahul S',  '2024-04-20', 'Basic',1200000,'At Risk'),
    (8,  'Kumar Properties',       'Pune',   'Priya M',  '2024-05-05', 'Pro',1800000,'Active'),
    (9,  'Eastern Group',          'Delhi',  'Vikram K', '2024-05-22', 'Basic',1200000,'At Risk'),
    (10, 'Kolte Patil',            'Pune',   'Priya M',  '2024-06-08', 'Pro',2400000,'Active'),
    (11, 'Runwal Group',           'Mumbai', 'Rahul S',  '2024-07-01', 'Basic',1200000,'Active'),
    (12, 'Pride World City',       'Pune',  'Vikram K', '2024-08-10', 'Pro',1800000,'At Risk'),
]
cur.execute("DELETE FROM clients")
cur.executemany("INSERT IGNORE INTO clients VALUES (%s,%s,%s,%s,%s,%s,%s,%s)", clients)

# ── 2. INSERT CALLS ───────────────────────────────
statuses      = ['Connected']*7 + ['Dropped']*2 + ['No_Answer']
ai_statuses   = ['Success']*75 + ['Failed']*25
start_date    = datetime(2024, 6, 1)

call_rows = []
for i in range(1, 1201):
    cid   = random.randint(1, 12)
    score = random.randint(10, 95)
    dur   = random.randint(30, 900)
    daysago = random.randint(0, 180)
    call_rows.append((
        i, cid, random.randint(1,500),
        random.choice(['Amit','Neha','Rohan','Sonam']),
        (start_date + timedelta(days=daysago)).strftime('%Y-%m-%d'),
        dur,
        random.choice(statuses),
        random.choice(ai_statuses),
        score
    ))
cur.executemany(
    "INSERT IGNORE INTO calls VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)",
    call_rows
)

# ── 3. INSERT USER LOGINS ─────────────────────────
login_rows = []
for i in range(1, 2001):
    cid    = random.randint(1, 12)
    # Some clients login less (churn risk)
    weight = 1 if cid in [3,9] else 5
    daysago = random.randint(1, 60) * weight
    login_rows.append((
        i, cid, random.randint(1,5),
        (datetime.now() - timedelta(days=daysago)).strftime('%Y-%m-%d %H:%M:%S')
    ))
cur.executemany(
    "INSERT IGNORE INTO user_logins VALUES (%s,%s,%s,%s)",
    login_rows
)

# ── 4. INSERT FEATURE LOGS ────────────────────────
modules  = ['PreSales']*6 + ['Marketing']*2 + ['PostSales']
features = {
    'PreSales':  ['Add Lead','Move Stage','View Summary','Set Follow-up'],
    'Marketing': ['Generate Brochure','Send WhatsApp','Create Creative'],
    'PostSales': ['Post Update','Payment Reminder','View Ticket']
}
feat_rows = []
for i in range(1, 3001):
    mod = random.choice(modules)
    cid = random.randint(1,12)
    daysago = random.randint(1,30)
    feat_rows.append((
        i, cid, random.randint(1,5), mod,
        random.choice(features[mod]),
        (datetime.now()-timedelta(days=daysago)).strftime('%Y-%m-%d %H:%M:%S')
    ))
cur.executemany(
    "INSERT IGNORE INTO feature_logs VALUES (%s,%s,%s,%s,%s,%s)",
    feat_rows
)

# ── 5. INSERT LEAD STAGE HISTORY ──────────────────
stages    = ['New Lead','Qualified','Site Visit','Opportunity','Closed']
stage_rows = []
row_id = 1
for lead_id in range(1, 301):
    cid     = random.randint(1,12)
    t       = datetime(2024,6,1) + timedelta(days=random.randint(0,150))
    n_stages = random.randint(2, 5)
    for s in stages[:n_stages]:
        stage_rows.append((row_id, lead_id, cid, s, t.strftime('%Y-%m-%d %H:%M:%S')))
        t += timedelta(days=random.randint(1,15))
        row_id += 1
cur.executemany(
    "INSERT IGNORE INTO lead_stage_history VALUES (%s,%s,%s,%s,%s)",
    stage_rows
)

# ── 6. INSERT TICKETS ─────────────────────────────
cats = ['Construction Update']*4+['Payment Confusion']*3+['App Not Loading']*2+['Document Download']
ticket_rows = []
for i in range(1, 221):
    cid     = random.randint(1,12)
    created = datetime.now() - timedelta(days=random.randint(1,60))
    resolved = created + timedelta(hours=random.randint(1,48)) if random.random()>0.15 else None
    ticket_rows.append((
        i, cid, random.randint(1,50),
        random.choice(cats),
        'Resolved' if resolved else 'Open',
        created.strftime('%Y-%m-%d %H:%M:%S'),
        resolved.strftime('%Y-%m-%d %H:%M:%S') if resolved else None
    ))
cur.executemany(
    "INSERT IGNORE INTO tickets VALUES (%s,%s,%s,%s,%s,%s,%s)",
    ticket_rows
)

conn.commit()
conn.close()
print("✅ All sample data created successfully!")
print("Clients: 12 | Calls: 1200 | Logins: 2000 | Features: 3000 | Stages: 900 | Tickets: 220")

✅ All sample data created successfully!
Clients: 12 | Calls: 1200 | Logins: 2000 | Features: 3000 | Stages: 900 | Tickets: 220
